# Gold: risky hosts per department

Key, sip, parameters and target come from `gold_risk.json`. Each `%%sql` cell that makes a view is a step; the rest is exploration and never runs in the pipeline.


In [ ]:
from duckduck.pipeline import notebook

nb = notebook("gold_risk.json", config_path="duckduck.pipeline.json")

In [ ]:
%%sql
CREATE VIEW exposed AS
SELECT s.*, o.owner, o.department
FROM lake_silver_assets s
LEFT JOIN owners o ON s.ip = o.ip
WHERE s.risk >= {{ min_risk }}

In [ ]:
%%sql
-- exploration: not a view, so the pipeline never runs it
SELECT department, count(*) FROM exposed GROUP BY ALL ORDER BY 2 DESC

In [ ]:
%%sql
CREATE VIEW risk_by_department AS
SELECT coalesce(department, 'unknown') AS department, os, count(*) AS hosts, round(avg(risk), 1) AS avg_risk
FROM exposed
GROUP BY ALL

In [ ]:
nb.sip  # every followed key so far